# Notebook 01 — Modelos Baseline

**Tech Challenge FIAP — Fase 2**

Este notebook reproduz os modelos da Fase 1 como **linha de base (baseline)** para comparação com os modelos otimizados via Algoritmo Genético (Notebook 02).

**Modelos treinados:**
- Regressão Logística
- Random Forest
- SVM (kernel RBF)

**Dataset:** Maternal Health Risk — UCI (~790 registros, 6 features clínicas)

**Métrica principal:** F1-macro (problema multiclasse com desbalanceamento moderado)

**Resultado esperado (Fase 1):** Random Forest — F1-macro ~0.90, ROC-AUC ~0.98

## 0. Imports e configurações

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Garante que src/ esteja no path ao rodar o notebook de dentro de notebooks/
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipelines.preprocessing import build_preprocessed_splits
from src.models.baseline import build_pipelines, run_cross_validation, run_grid_search, summarize_best_params
from src.evaluation.metrics import evaluate_model, build_comparison_table, plot_feature_importance

print(f"Projeto raiz: {PROJECT_ROOT}")

## 1. Pré-processamento e split treino/teste

In [ ]:
X_train, X_test, y_train, y_test, scaler = build_preprocessed_splits(
    add_pulse_pressure=False,
    add_age_features=False,
    convert_temp=True,
    scale=True,
)

print(f"\nShapes — X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Features: {X_train.columns.tolist()}")

## 2. Validação cruzada (k=5) — visão geral dos modelos

In [ ]:
pipelines = build_pipelines()

df_cv = run_cross_validation(pipelines, X_train, y_train)

print("\nResultados — Validação Cruzada (k=5):")
df_cv

In [ ]:
# Gráfico comparativo de F1-macro na CV
fig, ax = plt.subplots(figsize=(8, 4))
modelos = df_cv.index.tolist()
f1_medias = df_cv["f1_macro_media"].values
f1_stds = df_cv["f1_macro_std"].values

bars = ax.barh(modelos, f1_medias, xerr=f1_stds, color="steelblue", alpha=0.8, capsize=5)
ax.set_xlabel("F1-macro (média ± std)")
ax.set_title("Validação Cruzada — F1-macro por Modelo (Baseline)")
ax.set_xlim(0, 1.05)
for i, (v, e) in enumerate(zip(f1_medias, f1_stds)):
    ax.text(v + e + 0.01, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "figures" / "baseline_cv_f1_macro.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Otimização de hiperparâmetros — GridSearchCV (Baseline)

> **Nota:** Este GridSearch usa as grades originais da Fase 1 e serve como **ponto de referência** para o Algoritmo Genético do Notebook 02.

In [ ]:
grid_searches = run_grid_search(pipelines, X_train, y_train)

print("\nMelhores hiperparâmetros (GridSearch baseline):")
summarize_best_params(grid_searches)

## 4. Avaliação no conjunto de teste

In [ ]:
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

resultados = {}
for nome, gs in grid_searches.items():
    metricas = evaluate_model(
        nome=nome,
        pipeline=gs.best_estimator_,
        X_test=X_test,
        y_test=y_test,
        plot_confusion=True,
        save_path=str(FIGURES_DIR / f"baseline_confusion_{nome}.png"),
    )
    metricas["best_params"] = gs.best_params_
    resultados[nome] = metricas

In [ ]:
# Tabela comparativa final
df_resultado = build_comparison_table(resultados)
print("\nComparativo Final — Conjunto de Teste:")
df_resultado.drop(columns=["best_params"], errors="ignore")

## 5. Feature Importance — Random Forest

In [ ]:
rf_pipeline = grid_searches["random_forest"].best_estimator_

importancias = plot_feature_importance(
    pipeline=rf_pipeline,
    feature_names=X_train.columns.tolist(),
    save_path=str(FIGURES_DIR / "baseline_feature_importance_rf.png"),
)

print("\nFeature Importance (Random Forest):")
print(importancias)

## 6. Salvar resultados baseline

Esses resultados serão usados como referência no Notebook 02 (Algoritmo Genético) para comparação antes/depois da otimização.

In [ ]:
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

# Remove best_params do JSON de métricas (não serializável de forma limpa)
baseline_export = {}
for nome, metricas in resultados.items():
    baseline_export[nome] = {k: v for k, v in metricas.items() if k != "best_params"}
    baseline_export[nome]["best_params"] = {
        k: str(v) for k, v in metricas.get("best_params", {}).items()
    }

output_path = EXPERIMENTS_DIR / "baseline_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_export, f, indent=2, ensure_ascii=False)

print(f"Resultados baseline salvos em: {output_path}")
print(json.dumps(baseline_export, indent=2, ensure_ascii=False))

## 7. Conclusão

| Modelo | F1-macro | ROC-AUC | Recall High Risk |
|--------|----------|---------|------------------|
| Random Forest | — | — | — |
| SVM | — | — | — |
| Regressão Logística | — | — | — |

> Preencher após execução. Esses valores são o **ponto de partida** — o Algoritmo Genético (Notebook 02) vai tentar superá-los automaticamente.